<a href="https://colab.research.google.com/github/kulanigayathree/J26-IT-364/blob/model%2FFishing-Zone-Prediction/notebooks/C2_fishing_zone_prediction/C2_7_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# C2 – Notebook 7: Fishing-zone demo page (Gradio)

Pick a model (or the ensemble), a date and Day 1/2/3 → see the **probability map**, **uncertainty**,
**what actually happened**, and the **top fishing zones with latitude/longitude**.
By default it uses the best choice from Notebook 6 (`results/best_model.txt`).

`Runtime → Change runtime type → T4 GPU` (CPU also works, slower), then `Runtime → Run all`.
A link appears at the bottom; the `gradio.live` link works on any phone/laptop for 72 hours.

## Step 1 – Connect Drive and install Gradio

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
BASE = '/content/drive/MyDrive/C2_data'
CKPT_ROOT = f'{BASE}/checkpoints'; os.makedirs(CKPT_ROOT, exist_ok=True)
OUT = f'{BASE}/results';           os.makedirs(OUT, exist_ok=True)
LOCAL_CUBE = '/content/datacube_0p25.nc'
if not os.path.exists(LOCAL_CUBE):                 # copy once to Colab's fast disk (much quicker than reading Drive)
    shutil.copy(f'{BASE}/datacube_0p25.nc', LOCAL_CUBE)
print('Datacube ready:', LOCAL_CUBE)

!pip install -q gradio

Mounted at /content/drive
Datacube ready: /content/datacube_0p25.nc


## Step 2 – Settings and data (same as training)

In [2]:
import numpy as np, pandas as pd, xarray as xr, time, json, pickle, copy
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.isotonic import IsotonicRegression

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '(turn on the T4 GPU if this says cpu)')
torch.manual_seed(42); np.random.seed(42)

# ---- same for EVERY model (fair comparison) ----
T_IN, LEADS = 7, 3                     # 7 input days (ending yesterday) -> Day 1, Day 2, Day 3
# GFW coverage in this area rose sharply Dec 2021 - early 2022 (0.5% -> 7-14% fishing cells),
# so labels before Mar 2022 are not comparable: they are used ONLY for pre-training (below).
TRAIN    = ('2022-03-01', '2023-12-31')
VAL      = ('2024-01-01', '2024-12-31')
TEST     = ('2025-01-01', '2025-12-31')
PRETRAIN = ('2017-01-01', '2021-11-30')   # old low-coverage years: learn ocean patterns, then fine-tune
FEATURES = ['sst', 'sst_grad', 'log_chl', 'sla', 'u', 'v', 'cur_speed', 'mld',
            'depth', 'moon_sin', 'moon_cos', 'doy_sin', 'doy_cos',
            'fish_clim']               # usual fishing grounds (leave-one-year-out, see Step 3)

# ---- method switches (the comparison notebook reports what was used) ----
RESIDUAL     = True    # model learns CORRECTIONS on top of climatology (usual fishing grounds)
USE_PRETRAIN = False   # pre-train on PRETRAIN years, then fine-tune on TRAIN

EPOCHS, BATCH, LR, DROPOUT = 30, 16, 1e-3, 0.2
PATIENCE = 5                           # early stopping (fine-tuning / normal training)
PRE_EPOCHS, PRE_PATIENCE = 15, 3       # pre-training
FINETUNE_LR = 3e-4                     # smaller learning rate when fine-tuning a pre-trained model
MC_RUNS, CAL_RUNS = 30, 10             # MC dropout runs for test / for fitting calibration

Device: cpu (turn on the T4 GPU if this says cpu)


In [3]:
X = xr.open_dataset(LOCAL_CUBE).load()
days = pd.DatetimeIndex(X.time.values)
ocean = X['depth'].isel(time=0).notnull().values                   # (H, W) True = sea
label = np.nan_to_num(X['fishing'].values).astype('float32')       # (time, H, W) 1 = fishing
tr  = (days >= TRAIN[0]) & (days <= TRAIN[1])
pre = (days >= PRETRAIN[0]) & (days <= PRETRAIN[1])

# Usual fishing grounds = how often each cell was fished in this month.
#  * validation / test days: full training period
#  * training and pre-training days: LEAVE-ONE-YEAR-OUT (from the other years of the same period),
#    so the map never contains the answer for that day.
clim_m = np.stack([label[tr & (days.month == m)].mean(0) for m in range(1, 13)])     # (12, H, W)
fc = clim_m[np.asarray(days.month) - 1].copy()
def leave_one_year_out(period):
    for y in sorted(set(days[period].year)):
        other = period & (days.year != y)
        if not other.any(): continue
        other_all = label[other].mean(0)                   # fallback: other years, all months
        for m in range(1, 13):
            these = period & (days.year == y) & (days.month == m)
            if not these.any(): continue
            src = other & (days.month == m)
            fc[these] = label[src].mean(0) if src.any() else other_all
leave_one_year_out(tr); leave_one_year_out(pre)
X['fish_clim'] = (('time', 'latitude', 'longitude'), np.where(ocean, fc, np.nan).astype('float32'))

# Residual learning: the climatology of the TARGET day, as a logit offset added to the model output
pc = np.clip(fc, 1e-3, 1 - 1e-3)
OFF = np.where(ocean, np.log(pc / (1 - pc)), 0).astype('float32')

# Features -> (time, channel, H, W), normalised with TRAINING statistics only
feat = np.stack([X[v].values for v in FEATURES], axis=1).astype('float32')
mean = np.nanmean(feat[tr], axis=(0, 2, 3), keepdims=True)
std  = np.nanstd(feat[tr],  axis=(0, 2, 3), keepdims=True) + 1e-6
feat = np.nan_to_num((feat - mean) / std, nan=0.0)                 # land / gaps -> 0

NT, C, H, W = feat.shape
lat, lon = X.latitude.values, X.longitude.values
print(f'{NT} days, {C} input channels, grid {H}x{W}, sea cells {ocean.sum()}')
for nm, (a, b) in [('pretrain', PRETRAIN), ('train', TRAIN), ('val', VAL), ('test', TEST)]:
    sel = (days >= a) & (days <= b)
    print(f'Fishing rate {nm:8s} {a[:7]}..{b[:7]}:', round(float(label[sel][:, ocean].mean()), 4))
json.dump({'features': FEATURES, 'mean': mean.ravel().tolist(), 'std': std.ravel().tolist()},
          open(f'{OUT}/normalisation.json', 'w'))

3287 days, 14 input channels, grid 40x44, sea cells 1421
Fishing rate pretrain 2017-01..2021-11: 0.0045
Fishing rate train    2022-03..2023-12: 0.1127
Fishing rate val      2024-01..2024-12: 0.1387
Fishing rate test     2025-01..2025-12: 0.0725


## Step 3 – Load the trained models

In [4]:
def conv_block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                         nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU())

class UNet(nn.Module):
    # Space only: looks at the map of the last `use_days` day(s) with no memory of how it changed.
    # Encoder shrinks the map 40x44 -> 20x22 -> 10x11 to see big patterns, decoder grows it back.
    def __init__(self, cin, p=0.2, out=3, base=32, use_days=1):
        super().__init__(); self.use_days = use_days
        self.e1 = conv_block(cin * use_days, base)
        self.e2 = conv_block(base, base * 2)
        self.mid = conv_block(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2); self.d2 = conv_block(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2);     self.d1 = conv_block(base * 2, base)
        self.drop = nn.Dropout2d(p)
        self.head = nn.Conv2d(base, out, 1)
    def forward(self, x):                                  # x: (B, T, C, H, W)
        B, T, C, H, W = x.shape
        x = x[:, -self.use_days:].reshape(B, self.use_days * C, H, W)   # last day(s) stacked as channels
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        m = self.drop(self.mid(self.pool(e2)))
        d2 = self.drop(self.d2(torch.cat([self.up2(m), e2], 1)))
        d1 = self.d1(torch.cat([self.up1(d2), e1], 1))
        return self.head(self.drop(d1))

class ConvLSTMCell(nn.Module):
    def __init__(self, cin, ch, k=3):
        super().__init__(); self.ch = ch
        self.conv = nn.Conv2d(cin + ch, 4 * ch, k, padding=k // 2)
    def forward(self, x, h, c):
        i, f, o, g = torch.chunk(self.conv(torch.cat([x, h], 1)), 4, 1)
        c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
        return torch.sigmoid(o) * torch.tanh(c), c

class ConvLSTMNet(nn.Module):
    # Space + time: reads the 7 days in order, remembering what it has seen (cell state)
    def __init__(self, cin, p=0.2, out=3, hidden=(32, 32)):
        super().__init__()
        self.cells = nn.ModuleList(); prev = cin
        for ch in hidden:
            self.cells.append(ConvLSTMCell(prev, ch)); prev = ch
        self.drop = nn.Dropout2d(p)
        self.head = nn.Sequential(nn.Conv2d(prev, 32, 3, padding=1), nn.ReLU(),
                                  nn.Dropout2d(p), nn.Conv2d(32, out, 1))
    def forward(self, x):                                  # x: (B, T, C, H, W)
        B, T, _, H, W = x.shape
        state = [(x.new_zeros(B, c.ch, H, W), x.new_zeros(B, c.ch, H, W)) for c in self.cells]
        for t in range(T):
            inp = x[:, t]
            for j, cell in enumerate(self.cells):
                h, c = cell(inp, *state[j]); state[j] = (h, c)
                inp = self.drop(h) if j < len(self.cells) - 1 else h
        return self.head(self.drop(inp))                   # logits (B, LEADS, H, W)

class ConvGRUCell(nn.Module):
    def __init__(self, cin, ch, k=3):
        super().__init__(); self.ch = ch
        self.gates = nn.Conv2d(cin + ch, 2 * ch, k, padding=k // 2)   # update (z) and reset (r) gates
        self.cand  = nn.Conv2d(cin + ch, ch, k, padding=k // 2)
    def forward(self, x, h):
        z, r = torch.chunk(torch.sigmoid(self.gates(torch.cat([x, h], 1))), 2, 1)
        n = torch.tanh(self.cand(torch.cat([x, r * h], 1)))
        return (1 - z) * h + z * n

class ConvGRUNet(nn.Module):
    # Space + time, like ConvLSTM but a simpler memory (no separate cell state) -> fewer parameters
    def __init__(self, cin, p=0.2, out=3, hidden=(32, 32)):
        super().__init__()
        self.cells = nn.ModuleList(); prev = cin
        for ch in hidden:
            self.cells.append(ConvGRUCell(prev, ch)); prev = ch
        self.drop = nn.Dropout2d(p)
        self.head = nn.Sequential(nn.Conv2d(prev, 32, 3, padding=1), nn.ReLU(),
                                  nn.Dropout2d(p), nn.Conv2d(32, out, 1))
    def forward(self, x):                                  # x: (B, T, C, H, W)
        B, T, _, H, W = x.shape
        state = [x.new_zeros(B, c.ch, H, W) for c in self.cells]
        for t in range(T):
            inp = x[:, t]
            for j, cell in enumerate(self.cells):
                state[j] = cell(inp, state[j])
                inp = self.drop(state[j]) if j < len(self.cells) - 1 else state[j]
        return self.head(self.drop(inp))

CLASSES = {'unet': UNet, 'convlstm': ConvLSTMNet, 'convgru': ConvGRUNet}
LABELS = {'unet': 'U-Net (space only)', 'convlstm': 'ConvLSTM (space + time)', 'convgru': 'ConvGRU (space + time)'}
ENS = 'Ensemble (average of deep models)'

pos = label[tr][:, ocean].mean()
LOG_W = float(np.log(min((1 - pos) / max(pos, 1e-4), 50)))

loaded = {}
for key, Cls in CLASSES.items():
    ck, cf, cal = f'{CKPT_ROOT}/{key}/best.pt', f'{OUT}/{key}_config.json', f'{OUT}/{key}_calibrator.pkl'
    if not all(os.path.exists(f) for f in (ck, cf, cal)): continue
    cfg = json.load(open(cf))
    m = Cls(C, p=cfg['DROPOUT'], out=cfg['LEADS'], **cfg['model_args']).to(DEVICE)
    m.load_state_dict(torch.load(ck, map_location=DEVICE))
    loaded[LABELS[key]] = dict(model=m, calib=pickle.load(open(cal, 'rb')), t_in=cfg['T_IN'],
                               residual=cfg.get('RESIDUAL', False))
choices = list(loaded) + ([ENS] if len(loaded) >= 2 else [])
best_file = f'{OUT}/best_model.txt'
bk = open(best_file).read().strip() if os.path.exists(best_file) else ''
DEFAULT = ENS if bk == 'ensemble' and ENS in choices else LABELS.get(bk, choices[0])
print('Models ready:', choices, '| default:', DEFAULT)

Models ready: ['U-Net (space only)', 'ConvLSTM (space + time)', 'ConvGRU (space + time)', 'Ensemble (average of deep models)'] | default: Ensemble (average of deep models)


## Step 4 – Prediction function

In [5]:
def enable_mc(m):
    m.eval()
    for layer in m.modules():
        if isinstance(layer, (nn.Dropout, nn.Dropout2d)): layer.train()

@torch.no_grad()
def member_runs(info, t, lead, runs):
    m, t_in = info['model'], info['t_in']
    if t < t_in - 1: raise ValueError(f'Date too early: the model needs {t_in} earlier days')
    x = torch.from_numpy(feat[t - t_in + 1: t + 1]).unsqueeze(0).to(DEVICE)
    off = torch.from_numpy(OFF[t + 1: t + 1 + LEADS]).unsqueeze(0).to(DEVICE)
    enable_mc(m)
    ps = []
    for _ in range(int(runs)):
        lo = m(x) + off if info['residual'] else m(x)
        ps.append(torch.sigmoid(lo - LOG_W)[0, lead - 1].cpu().numpy())
    ps = np.stack(ps)
    return info['calib'][lead - 1].predict(ps.ravel()).reshape(ps.shape)

def forecast(model_label, target_date, lead_day, mc_runs):
    try: target = pd.Timestamp(str(target_date)[:10])
    except Exception: raise ValueError('Enter a date like 2025-06-15')
    lead = int(str(lead_day).split()[1])
    if target not in days: raise ValueError(f'Date must be between {days[0].date()} and {days[-1].date()}')
    t = days.get_loc(target) - lead                       # last input day ("yesterday" when lead = 1)
    members = list(loaded.values()) if model_label == ENS else [loaded[model_label]]
    ps = np.concatenate([member_runs(info, t, lead, mc_runs) for info in members])   # all MC runs of all members
    return ps.mean(0), ps.std(0), label[t + lead], t

def sea(a): return np.where(ocean, a, np.nan)

def top_zones(p, u, n=10):
    yy, xx = np.where(ocean)
    order = np.argsort(-p[yy, xx])[:n]
    return pd.DataFrame({'Rank': np.arange(1, len(order) + 1),
                         'Latitude (°N)': lat[yy[order]].round(3), 'Longitude (°E)': lon[xx[order]].round(3),
                         'Fishing probability': p[yy[order], xx[order]].round(3),
                         'Uncertainty': u[yy[order], xx[order]].round(3)})

def run_demo(model_label, target_date, lead_day, mc_runs):
    p, u, y, t = forecast(model_label, target_date, lead_day, mc_runs)
    fig, ax = plt.subplots(1, 4, figsize=(20, 4.8))
    panels = [(sea(p), 'Fishing probability', 'viridis', (0, max(0.05, np.nanmax(sea(p))))),
              (sea(u), 'Uncertainty (MC std)', 'magma', None),
              (sea(y), 'Actual fishing (GFW)', 'Greys', (0, 1)),
              (sea(X['sst'].values[t]), f'SST on last input day {days[t].date()} (°C)', 'coolwarm', None)]
    for a, (arr, title, cmap, lim) in zip(ax, panels):
        kw = {} if lim is None else {'vmin': lim[0], 'vmax': lim[1]}
        im = a.pcolormesh(lon, lat, arr, cmap=cmap, shading='auto', **kw)
        a.set_title(title, fontsize=11); a.set_aspect('equal'); plt.colorbar(im, ax=a, shrink=0.75)
    zones = top_zones(p, u)
    ax[0].scatter(zones['Longitude (°E)'], zones['Latitude (°N)'], s=40, facecolors='none', edgecolors='red', lw=1.5)
    fig.suptitle(f'{model_label} | forecast for {pd.Timestamp(str(target_date)[:10]).date()}'
                 f' | {lead_day} | inputs up to {days[t].date()}', fontsize=13)
    plt.tight_layout()
    yy, pp = y[ocean], p[ocean]
    top = pp >= np.quantile(pp, 0.9)
    stats = {'Sea cells with real fishing': int(yy.sum()),
             'Average probability': round(float(pp.mean()), 4),
             'Real fishing rate (all sea cells)': round(float(yy.mean()), 4),
             'Fishing rate in the top-10% cells': round(float(yy[top].mean()), 4),
             'Average uncertainty': round(float(u[ocean].mean()), 4),
             'ROC-AUC for this day': round(float(roc_auc_score(yy, pp)), 4) if 0 < yy.sum() < len(yy) else 'n/a'}
    return fig, zones, pd.DataFrame(stats.items(), columns=['Measure', 'Value'])

## Step 5 – Launch the demo page
Red circles on the probability map = the top 10 zones listed in the table.

In [6]:
import gradio as gr
with gr.Blocks(title='C2 Fishing-zone forecast') as demo:
    gr.Markdown('## C2 – Fishing-zone forecast (next 72 h)\n'
                'Test period 2025. Day 1 = the day after the last input day. '
                'Probabilities are model estimates of fishing activity, not a guarantee of catch.')
    with gr.Row():
        model_in = gr.Dropdown(choices, value=DEFAULT, label='Model')
        date_in = gr.Textbox(label='Forecast date (YYYY-MM-DD)', value='2025-06-15')
        lead_in = gr.Radio(['Day 1', 'Day 2', 'Day 3'], value='Day 1', label='How many days ahead')
        runs_in = gr.Slider(5, 50, value=30, step=5, label='MC dropout runs (per model)')
    btn = gr.Button('Predict', variant='primary')
    plot_out = gr.Plot(label='Maps')
    zones_out = gr.Dataframe(label='Top 10 fishing zones (latitude / longitude)')
    table_out = gr.Dataframe(label='How did the model do on this day?')
    btn.click(run_demo, [model_in, date_in, lead_in, runs_in], [plot_out, zones_out, table_out])

demo.launch(share=True, show_error=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e704aa1c9c77a6f8c9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
